In [19]:
from nn_accelerated import init_model, ModelMetrics, sep_labels, to_device_batches, run_all_epochs, run_epoch, train_step
import numpy as np
from jax import random

In [20]:
n_input = 784
n_hidden = 30
n_output = 10
classes = 10

d_train = sep_labels(np.loadtxt("/home/roman/learning-ml/2802ict-assignments/assignment_02/nn/fashion-mnist_train.csv.gz", delimiter=',', skiprows=1), classes) # (60000, 785) first col is label
d_test = sep_labels(np.loadtxt("/home/roman/learning-ml/2802ict-assignments/assignment_02/nn/fashion-mnist_test.csv.gz", delimiter=',', skiprows=1), classes) # (10000, 785) first col is label

batched_d_train = to_device_batches(d_train[0], d_train[1], 32)
batched_d_test = to_device_batches(d_test[0], d_test[1], 32)

key = random.key(0)

In [22]:
model = init_model(key, n_input, n_hidden, n_output)
metrics = ModelMetrics()

lr = 3
n_epochs = 30
batch_size = 20

## We run the model once to make sure JAX jits everything before we benchmark
run_all_epochs(model, metrics, batched_d_train[0], batched_d_train[1], n_epochs, lr)

print("Model ready!")

Model ready!


Benchmarking the epoch-vectorised approach:

In [15]:
%%timeit -n5 -r2

run_all_epochs(model, metrics, batched_d_train[0], batched_d_train[1], n_epochs, lr)

1.57 s ± 30.7 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


Benchmarking the per-epoch vectorised approach:

In [14]:
%%timeit -n5 -r2

def bench(model, metrics):
    for i in range(n_epochs):
        model, metrics = run_epoch(model, metrics, batched_d_train[0], batched_d_train[1], lr)

bench(model, metrics)
    

1.54 s ± 26.3 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)


Benchmarking the per-minibatch vectorised approach:

In [18]:
%%timeit -n5 -r2

def bench(model, metrics):
    for i in range(n_epochs):
        for k in range(batched_d_train[0].shape[0]):
            model, metrics = train_step((batched_d_train[0][k], batched_d_train[1][k]), model, metrics, lr)

bench(model, metrics)

1min 16s ± 627 ms per loop (mean ± std. dev. of 2 runs, 5 loops each)
